In [1]:
import torch
import torch.nn as nn
from torch.autograd import Variable
import numpy as np
from torch.distributions import Normal, Independent, kl
import torchvision
import torchvision.transforms as T
from torch.distributions import Normal, Independent, kl, MultivariateNormal
from torch.utils.tensorboard import SummaryWriter
from torchmetrics import ConfusionMatrix
from ignite.metrics import *

In [2]:
# default `log_dir` is "runs" - we'll be more specific here
#writer = SummaryWriter('runs/fashion_mnist_experiment_1')

In [3]:
class ResidualBlock(nn.Module):
    def __init__(self, in_channels, stride=1):
        super(ResidualBlock, self).__init__()
        self.conv1 = nn.Conv2d(in_channels, in_channels, kernel_size=3, stride=stride, padding=1, bias=False)
        self.bn1 = nn.BatchNorm2d(in_channels)
        self.relu = nn.ReLU(inplace=True)
        self.conv2 = nn.Conv2d(in_channels, in_channels, kernel_size=3, stride=stride, padding=1, bias=False)
        self.bn2 = nn.BatchNorm2d(in_channels)
        
    def forward(self, x):
        residual = x
        out = self.conv1(x)
        out = self.bn1(out)
        out = self.relu(out)
        out = self.conv2(out)
        out = self.bn2(out)
        out += residual
        out = self.relu(out)
        return out

In [4]:
class DownConvBlock(nn.Module):
    """
    A block of three convolutional layers where each layer is followed by a non-linear activation function
    Between each block we add a pooling operation.
    """
    def __init__(self, input_dim, output_dim, initializers, padding, latent_dim = None, ResLayers = 2):
        super(DownConvBlock, self).__init__()
        Reslayers = []
        self.latent_dim = latent_dim
        
        self.firstLayer = nn.Conv2d(input_dim, output_dim, kernel_size = 3, stride = 2, padding = int(padding))
        self.relu = nn.ReLU()
        for _ in range(ResLayers):
            Reslayers.append(ResidualBlock(output_dim))
        
        self.layers = nn.Sequential(*Reslayers)
        
        if self.latent_dim:
            self.distLayer = nn.Conv2d(output_dim, 2 * self.latent_dim, (1,1), stride=1)
    
    def forward(self, inputFeatures):
        
        emb = self.relu(self.firstLayer(inputFeatures))
        out = self.layers(emb)
        
        if self.latent_dim:
            
            mu_log_sigma = self.distLayer(out)
            mu_log_sigma = torch.squeeze(torch.squeeze(mu_log_sigma, dim=2), dim=2)
            mu = mu_log_sigma[:,:self.latent_dim]
            log_sigma = mu_log_sigma[:,self.latent_dim:]
            dist = Independent(Normal(loc=mu, scale=torch.exp(log_sigma)),1)
            
            return out, dist
        
        return out    
    

In [5]:
class UpConvBlock(nn.Module):
    """
    A block consists of an upsampling layer followed by a convolutional layer to reduce the amount of channels and then a DownConvBlock
    If bilinear is set to false, we do a transposed convolution instead of upsampling
    """
    def __init__(self, input_dim, output_dim, initializers, padding, latent_dim = None, ResLayers = 2):
        super(UpConvBlock, self).__init__()
        Reslayers = []
        self.latent_dim = latent_dim
        
        self.firstLayer = nn.ConvTranspose2d(input_dim, output_dim, kernel_size = 4, stride = 2, padding = int(padding))
        
        for _ in range(ResLayers):
            Reslayers.append(ResidualBlock(output_dim))
        
        self.layers = nn.Sequential(*Reslayers)
        
        if self.latent_dim:
            self.distLayer = nn.Conv2d(output_dim, 2 * self.latent_dim, (1,1), stride=1)

    def forward(self, inputFeatures):
            
        emb = self.firstLayer(inputFeatures)
        out = self.layers(emb)
        
        if self.latent_dim:
            
            mu_log_sigma = self.distLayer(out)
            mu_log_sigma = torch.squeeze(torch.squeeze(mu_log_sigma, dim=2), dim=2)
            mu = mu_log_sigma[:,:self.latent_dim]
            log_sigma = mu_log_sigma[:,self.latent_dim:]
            dist = Independent(Normal(loc=mu, scale=torch.exp(log_sigma)),1)
            
            return out, dist
            
        return out


In [6]:
class UNet(nn.Module):
    """
    A block consists of an upsampling layer followed by a convolutional layer to reduce the amount of channels and then a DownConvBlock
    If bilinear is set to false, we do a transposed convolution instead of upsampling
    """
    def __init__(self, LatentVarSize = 6, posterior = False):
        super(UNet, self).__init__()
        #Vars init
        self.LatentVarSize = LatentVarSize
        self.posterior = posterior

        
        #architecture
        self.DownConvBlock1 = DownConvBlock(input_dim = 6 if posterior else 3, output_dim = 128, ResLayers = 2, initializers = None, padding = 1)
        self.DownConvBlock2 = DownConvBlock(input_dim = 128, output_dim = 256, ResLayers = 2, initializers = None, padding = 1)
        self.DownConvBlock3 = DownConvBlock(input_dim = 256, output_dim = 512, ResLayers = 2, initializers = None, padding = 1)
        self.DownConvBlock4 = DownConvBlock(input_dim = 512, output_dim = 256, ResLayers = 2, latent_dim = LatentVarSize, initializers = None, padding = 1)

        self.UpConvBlock1 = UpConvBlock(input_dim = 256, output_dim = 512, ResLayers = 2, initializers = None, padding = 1)
        self.UpConvBlock2 = UpConvBlock(input_dim = (512 * 2) + LatentVarSize, output_dim = 256, ResLayers = 2, latent_dim = LatentVarSize, initializers = None, padding = 1)
        self.UpConvBlock3 = UpConvBlock(input_dim = (256 * 2) + LatentVarSize, output_dim = 128, ResLayers = 2, latent_dim = LatentVarSize, initializers = None, padding = 1)
        self.UpConvBlock4 = UpConvBlock(input_dim = (128 * 2) + LatentVarSize, output_dim = 32, ResLayers = 2, initializers = None, padding = 1)

        #loss functions
        self.criterion = nn.BCEWithLogitsLoss(size_average = False, reduction = None, reduce = False)
        
    def forward(self, inputFeatures):
        
        dists = {}
        encoderOuts = {}
        
        encoderOuts["out1"] = self.DownConvBlock1(inputFeatures)
        encoderOuts["out2"] = self.DownConvBlock2(encoderOuts["out1"])
        encoderOuts["out3"] = self.DownConvBlock3(encoderOuts["out2"])
        encoderOuts["out4"], dists["dist1"] = self.DownConvBlock4(encoderOuts["out3"])

        out = self.UpConvBlock1(encoderOuts["out4"])
        latent1 = torch.nn.Upsample(size=encoderOuts["out3"].shape[2:], mode='nearest')(dists["dist1"].rsample())
        out = torch.cat((encoderOuts["out3"], out, latent1), 1)

        out, dists["dist2"] = self.UpConvBlock2(out)
        latent2 = torch.nn.Upsample(size=encoderOuts["out2"].shape[2:], mode='nearest')(dists["dist2"].rsample())
        out = torch.cat((encoderOuts["out2"], out, latent2), 1)

        out, dists["dist3"] = self.UpConvBlock3(out)
        latent3 =  torch.nn.Upsample(size=encoderOuts["out1"].shape[2:], mode='nearest')(dists["dist3"].rsample())
        out = torch.cat((encoderOuts["out1"], out, latent3), 1)
                
        if self.posterior:
            return dists
        
        out = self.UpConvBlock4(out)
        return out, dists
    
    def inference(self, inputFeatures):
        
        with torch.no_grad():
            dists = {}
            encoderOuts = {}
            samples = []

            encoderOuts["out1"] = self.DownConvBlock1(inputFeatures)
            encoderOuts["out2"] = self.DownConvBlock2(encoderOuts["out1"])
            encoderOuts["out3"] = self.DownConvBlock3(encoderOuts["out2"])
            encoderOuts["out4"], dists["dist1"] = self.DownConvBlock4(encoderOuts["out3"])

            for _ in range(16):

                out = self.UpConvBlock1(encoderOuts["out4"])
                latent1 = torch.nn.Upsample(size=encoderOuts["out3"].shape[2:], mode='nearest')(dists["dist1"].rsample())
                out = torch.cat((encoderOuts["out3"], out, latent1), 1)
                
                if "dist2" not in dists.keys():
                    out, dists["dist2"] = self.UpConvBlock2(out)
                else:
                    out, _ = self.UpConvBlock2(out)
                latent2 = torch.nn.Upsample(size=encoderOuts["out2"].shape[2:], mode='nearest')(dists["dist2"].rsample())
                out = torch.cat((encoderOuts["out2"], out, latent2), 1)

                
                if "dist3" not in dists.keys():
                    out, dists["dist3"] = self.UpConvBlock3(out)
                else:
                    out, _ = self.UpConvBlock3(out)
                latent3 =  torch.nn.Upsample(size=encoderOuts["out1"].shape[2:], mode='nearest')(dists["dist3"].rsample())
                out = torch.cat((encoderOuts["out1"], out, latent3), 1)

                out = self.UpConvBlock4(out)

                samples.append(out)
        
        
        return torch.stack(samples), dists


In [7]:
class ProUNet(nn.Module):
    """
    A block consists of an upsampling layer followed by a convolutional layer to reduce the amount of channels and then a DownConvBlock
    If bilinear is set to false, we do a transposed convolution instead of upsampling
    """
    def __init__(self, LatentVarSize = 6, beta = 5., training = True):
        super(ProUNet, self).__init__()
        #Vars init
        self.LatentVarSize = LatentVarSize
        self.beta = beta
        self.training = training
        
        #architecture
        self.unet = UNet()
        if training:
            self.posterior = UNet(posterior = True)
        
        #loss functions
        self.criterion = nn.BCEWithLogitsLoss(size_average = False, reduction = None, reduce = False)
        
    def forward(self, inputImg, segmasks = None):
        
        seg, priorDists = self.unet(inputImg)
        
        if self.training:
            posteriorDists = self.posterior(torch.cat((inputImg, segmasks), 1))
            return seg, priorDists, posteriorDists
        
        return seg, priorDists
        
    def inference(self, inputFeatures):
        with torch.no_grad():
            return self.unet.inference(inputFeatures)
    
    
    def evaluation(self, inputFeatures, segmasks):
        
        with torch.no_grad():
            samples, priors = self.unet.inference(inputFeatures)
            posteriorDists = self.posterior(torch.cat((inputFeatures, segmasks), 1))
            return samples, priors, posteriorDists
    
    
    def rec_loss(self, img, seg):
        return self.criterion(input = img, target = seg)
    
    
    def kl_loss(self, priors, posteriors):
        
        klLoss = {}
        for level, (posterior, prior) in enumerate(zip(posteriors.items(), priors.items())):
            klLoss[level] = torch.mean(kl.kl_divergence(posterior[1], prior[1]), (1,2))
        return klLoss
    
    
    def elbo_loss(self, img, seg, priors, posteriors):
        
        rec_loss = torch.sum(torch.mean(self.rec_loss(img, seg),(2,3)),1)
        kl_losses = self.kl_loss(priors, posteriors)
        kl_mean = torch.mean(torch.stack([i for i in kl_losses.values()]), 0)
        
        loss = torch.mean(rec_loss + self.beta * kl_mean)
        
        return loss, torch.mean(kl_mean), kl_losses, torch.mean(rec_loss)

In [8]:
input_test1 = torch.rand((2,3,512,512))
seg_test1 = torch.rand((2,3,512,512))
seg_test2 = torch.rand((2,32,512,512))
model = ProUNet()
optimizer = torch.optim.Adam(model.parameters(), lr =  0.00001, weight_decay = 0.00001)

/home/lunet/wsmo6/.conda/envs/3.6/lib/python3.6/site-packages/torch/nn/_reduction.py:42: UserWarning: size_average and reduce args will be deprecated, please use reduction='none' instead.
  warnings.warn(warning.format(ret))


In [9]:
seg, priorDists, posteriorDists = model(input_test1,seg_test1)

In [10]:
samples, priors, posteriorDists = model.evaluation(input_test1,seg_test1)

In [25]:
loss, kl_mean, kl_losses, rec_loss = model.elbo_loss(input_test1, samples[0], priors, posteriorDists)

loss, kl_mean, kl_losses, rec_loss = model.elbo_loss(batch['image'].to(device), batch['seg'].to(device), priorDists, posteriorDists)

In [12]:
loss = model.elbo_loss(input_test1, seg_test1, priorDists, posteriorDists)
loss

(tensor(3626.9429, grad_fn=<MeanBackward0>),
 tensor(724.9482, grad_fn=<MeanBackward0>),
 {0: tensor([219.9494, 246.5693], grad_fn=<MeanBackward1>),
  1: tensor([614.7817, 624.9163], grad_fn=<MeanBackward1>),
  2: tensor([1329.1794, 1314.2931], grad_fn=<MeanBackward1>)},
 tensor(2.2018))

In [13]:
loss.backward()
optimizer.step()

AttributeError: 'tuple' object has no attribute 'backward'

In [39]:
from torch.utils.data import DataLoader
from dataloader_cityscapes import *
kwargs = {}
kwargs["data_path"] = "../datasets/augmented_cityscapes"
kwargs["batch_size"] = 32
img_w = 256
img_h = 256

preprocess_in = transforms.Compose([
transforms.ToTensor(),
transforms.Normalize(mean = [0.485, 0.456, 0.406], std = [0.229, 0.224, 0.225]),
transforms.Resize((256,256))
])

preprocess_ou = transforms.Compose([
transforms.ToTensor(),
transforms.Resize((256,256))
])

In [40]:
tr_loader = CityscapesLoader(dataset_path = kwargs["data_path"], transform_in = preprocess_in, transform_ou = preprocess_ou, mode = 'train')
train_loader = DataLoader(dataset = tr_loader, batch_size = kwargs["batch_size"], shuffle = True, drop_last = True)
kwargs["no_classes"] = tr_loader.get_num_classes()

val_loader = CityscapesLoader(dataset_path = kwargs["data_path"], transform_in = preprocess_in, transform_ou = preprocess_ou, mode = 'val')
val_loader = DataLoader(dataset = val_loader, batch_size = kwargs["batch_size"], shuffle = True, drop_last = True)

In [41]:
len(train_loader)

1022

In [42]:
train_loader[:10]

TypeError: 'DataLoader' object is not subscriptable

In [43]:
torch.cuda.device_count()

2

In [44]:
torch.cuda.current_device()

0

In [46]:
torch.cuda.get_device_name(1)

'NVIDIA RTX A6000'